# Milestone 7 - frame-to-frame stability

**Inputs to attach:** the WFLW dataset and the run output holding the baseline weights, the same as milestone 6. CPU accelerator.

**Read the sections in order, because the order is the finding.** Milestone 6 established that the pipeline is dominated by acquisition rather than by the landmark model, and that 86% of the remaining centre error is the Haar box landing differently on each face. Section 1 measures whether that carries over to time. If box jitter is large, every landmark number below it is mostly measuring the box, and the model's own contribution is the `ours_gt_box` row rather than the deployed one.

Sequences are synthesised from stills: a smooth random walk plus sensor noise, with the annotation warped alongside so ground truth is known per frame. A 2D warp cannot produce out-of-plane rotation, blinking or expression, so **these numbers are a lower bound on jitter**. Pass `--video` with a real clip for the self-consistency estimator, which measures a different thing and is reported separately.

In [ ]:
!rm -rf dms-layer1
!git clone -b claude/facial-landmark-perception-q80yhn https://github.com/keerthanpragnay1728-prog/dms-layer1.git
%cd dms-layer1
# provenance: confirm the commit this run uses BEFORE trusting any number
!git log --oneline -1
!pip install -q -r requirements.txt


In [ ]:
!python tests/run_tests.py

In [ ]:
# 30 sequences x 60 frames. Raise --sequences for tighter medians; the
# Haar pass is the cost, at roughly a third of a second per frame.
!python scripts/stability_harness.py --config configs/layer1_base.yaml --split test


In [ ]:
import yaml
from pathlib import Path
r = yaml.safe_load(Path('/kaggle/working/m7_stability/m7_stability.yaml').read_text())
print('box centre jitter (box-side units):', r['box']['centre_jitter'])
print('landmark jitter, % of IOD:')
for k, v in r['landmark_jitter_pct'].items():
    print(f'  {k:<14} {v:.3f}   dropout {r["dropout_pct"][k]:.1f}%')
print('pupil jitter by row:')
for k, v in r['group_jitter_pct'].items():
    print(f'  {k:<14} {v["pupils"]:.3f}')
